## Porfilio Performance

Use this notebook to calculate your performance compared to major indices

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import Lin

In [17]:
import os

# Get the current working directory
current_directory = os.getcwd()
print(f"Current Working Directory: {current_directory}")



Current Working Directory: /mnt/batch/tasks/shared/LS_root/mounts/clusters/spectral-nature3/code


In [18]:
# Change the working directory
new_directory = '/mnt/batch/tasks/shared/LS_root/mounts/clusters/spectral-nature3/code/Users/omai.r/spectral_nature/src'
os.chdir(new_directory)
print(f"Changed Working Directory to: {new_directory}")

Changed Working Directory to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/spectral-nature3/code/Users/omai.r/spectral_nature/src


In [5]:
DATA_PATH = '/mnt/batch/tasks/shared/LS_root/mounts/clusters/spectral-nature3/code/Users/omai.r/spectral_nature/data'
if os.path.exists(DATA_PATH):
    print(f"Data path {DATA_PATH} is valid.")
else:
    print(f"Data path {DATA_PATH} is not valid.")

Data path /mnt/batch/tasks/shared/LS_root/mounts/clusters/spectral-nature3/code/Users/omai.r/spectral_nature/data is valid.


In [2]:
import LinkedAuth
token = LinkedAuth.get_creds("spectral-nature-kvault", retreive = ['SimFinAPI'])


In [ ]:
import simfin as sf
sf.set_api_key('')
sf.set_data_dir('../data/stock_fundamental/')

In [19]:
import os
os.listdir('./')

['.amlignore',
 '.amlignore.amltmp',
 '.DS_Store',
 'alpaca-test.ipynb',
 'analysis_modules',
 'CurrentStatus.ipynb',
 'CurrentStatus.py',
 'data_modules',
 'fred',
 'front_end_app',
 'FundamentalAnalyzer.ipynb',
 'FundamentalAnalyzer.py',
 'GetPortfolioTest.ipynb',
 'GT_geopolitics.py',
 'GT_tradewar.py',
 'Holdings.ipynb',
 'LinkedAuth.py',
 'MarketExplorer.ipynb',
 'MarketExplorer.py',
 'MarketExplorerDataCruncher.py',
 'OptionFinder.ipynb',
 'OptionFinder.py',
 'PortfolioPerf.ipynb',
 'TechnicalAnalyzer.ipynb',
 'TechnicalAnalyzer.py',
 'trigger.sh',
 'utils',
 '__init__.py',
 '__pycache__']

In [ ]:
# ...existing code...

import os
import sys
from pathlib import Path

# Ensure we can import FundamentalAnalyzer.py regardless of CWD
repo_root = Path("/home/azureuser/cloudfiles/code")
fa_path = repo_root / "Users" / "omai.r" / "spectral_nature" / "src"
if str(fa_path) not in sys.path:
    sys.path.insert(0, str(fa_path))

import FundamentalAnalyzer as FA  # uses the module you showed

# ---- Helper to plot all 3 statements over time for a given stock ----

def plot_financial_statements_over_time(ticker: str, since: str | None = "2018Q1"):
    """
    Use FundamentalAnalyzer to plot income, balance sheet, and cash flow
    statements over time for a single ticker.
    """
    ticker = ticker.upper()
    tidy, pivots, errs = FA.run_quarterly_comparison(
        tickers=[ticker],
        since=since,          # e.g., '2018Q1' or None for full history
        api_key=None,         # FundamentalAnalyzer handles LinkedAuth/SimFin setup
        data_dir='../../data/stock_fundamental/',
        plot=True             # let FundamentalAnalyzer create the plots
    )

    if errs:
        print("Errors:", errs)
    return tidy, pivots

# Example usage: plots 3 financial statements for AAPL
tidy, pivots = plot_financial_statements_over_time("NVDA", since="2018Q1")

Dataset "us-income-quarterly" not on disk.


- Downloading ... 100.0%
- Extracting zip-file ... Done!
- Loading from disk ... Done!
Dataset "us-balance-quarterly" not on disk.
- Downloading ... 100.0%
- Extracting zip-file ... 

In [1]:
# Recreate STX only-up-to-2024 issue using FundamentalAnalyzer.py

import os
import sys
import importlib.util
import types
from pathlib import Path
from datetime import datetime
import pandas as pd
from IPython.display import display

# Optional: Provide SimFin API key via env var if LinkedAuth is unavailable
# os.environ["SIMFIN_API_KEY"] = "<your-simfin-api-key>"

# Ensure we can import FundamentalAnalyzer.py
repo_root = Path("/home/azureuser/cloudfiles/code")
candidate = repo_root / "Users" / "omai.r" / "spectral_nature" / "src" / "FundamentalAnalyzer.py"
if candidate.exists():
    sys.path.insert(0, str(candidate.parent))
else:
    # Fallback: search for the file under the workspace
    for p in repo_root.rglob("FundamentalAnalyzer.py"):
        sys.path.insert(0, str(p.parent))
        break

# If LinkedAuth is not installed, stub it to read SIMFIN_API_KEY from env
if importlib.util.find_spec("LinkedAuth") is None:
    la = types.ModuleType("LinkedAuth")
    def get_creds(vault, retreive=None):
        # Returns a list to match usage in FundamentalAnalyzer
        return [os.getenv("SIMFIN_API_KEY", "")]
    la.get_creds = get_creds
    sys.modules["LinkedAuth"] = la

import FundamentalAnalyzer as FA  # noqa: E402

ticker = "STX"

def summarize_tidy(df: pd.DataFrame, label: str):
    print(f"\n===== {label} (tidy) =====")
    if df is None or df.empty:
        print("Empty.")
        return
    df = df.copy()
    df["Report Date"] = pd.to_datetime(df["Report Date"], errors="coerce")
    max_dt = df["Report Date"].max()
    max_year = int(df["Report Date"].dt.year.max()) if pd.notna(max_dt) else None
    print(f"Rows: {len(df)} | Max Report Date: {max_dt} | Max Year: {max_year}")
    last_yqs = (
        df.sort_values("Report Date")["YearQ"].dropna().astype(str).unique()[-6:]
        if "YearQ" in df.columns else []
    )
    print("Last YearQ values:", ", ".join(last_yqs))
    # Show last few rows
    cols = [c for c in ["Report Date", "YearQ", "Ticker", "Metric", "Value"] if c in df.columns]
    display(df.sort_values("Report Date").tail(10)[cols].reset_index(drop=True))

def summarize_raw(df: pd.DataFrame, label: str):
    print(f"\n----- {label} (raw SimFin subset for {ticker}) -----")
    try:
        sub = FA._extract_stmt_for_ticker(df, ticker)  # uses module helper
    except Exception as e:
        print(f"Extract error: {e}")
        return
    base = sub.reset_index()
    if "Report Date" not in base.columns:
        print("No 'Report Date' column in raw subset.")
        display(base.head())
        return
    base["Report Date"] = pd.to_datetime(base["Report Date"], errors="coerce")
    qmask = base.get("Fiscal Period", pd.Series(index=base.index, dtype=object)).astype(str).isin(["Q1","Q2","Q3","Q4"])
    max_dt_all = base["Report Date"].max()
    max_dt_quarters = base[qmask]["Report Date"].max() if "Fiscal Period" in base.columns else None
    print(f"Rows (all): {len(base)} | Max Report Date (all): {max_dt_all}")
    if "Fiscal Period" in base.columns:
        print(f"Rows (quarters only): {qmask.sum()} | Max Report Date (quarters): {max_dt_quarters}")
        tail_cols = [c for c in ["Report Date","Fiscal Year","Fiscal Period"] if c in base.columns]
        display(base[qmask].sort_values("Report Date").tail(6)[tail_cols].reset_index(drop=True))
    else:
        display(base.sort_values("Report Date").tail(6)[["Report Date"]].reset_index(drop=True))

# Run the tidy pipeline without plotting to reproduce the cutoff
print("Running run_quarterly_comparison for STX (since=None, plot=False)...")
try:
    tidy, pivots, errs = FA.run_quarterly_comparison(
        tickers=[ticker],
        since=None,          # include full history
        api_key=None,        # FundamentalAnalyzer handles LinkedAuth
        data_dir=None,       # use module default
        plot=False           # no plots in notebook
    )
except Exception as e:
    print("run_quarterly_comparison failed:", e)
    tidy, pivots, errs = ({'income': pd.DataFrame(), 'balance': pd.DataFrame(), 'cashflow': pd.DataFrame()}, {}, {"__pipeline__": str(e)})

if errs:
    print("\nErrors captured:", errs)

summarize_tidy(tidy.get("income"),   "Income Statement")
summarize_tidy(tidy.get("balance"),  "Balance Sheet")
summarize_tidy(tidy.get("cashflow"), "Cash Flow")

# Compare with raw SimFin frames to see if cutoff originates pre/post shaping
print("\nLoading raw SimFin quarterly frames to compare...")
try:
    FA.setup_simfin()  # ensures API key and data dir
    inc_raw, bal_raw, cfs_raw = FA.load_quarterly_frames()
    summarize_raw(inc_raw, "Income (raw)")
    summarize_raw(bal_raw, "Balance (raw)")
    summarize_raw(cfs_raw, "Cash Flow (raw)")
except Exception as e:
    print("Raw frame load failed:", e)

# Simple reproduction flag: did all tidy frames cap at <= 2024?
def max_year(df):
    if df is None or df.empty:
        return None
    dts = pd.to_datetime(df["Report Date"], errors="coerce").dropna()
    return int(dts.dt.year.max()) if not dts.empty else None

years = {k: max_year(v) for k, v in tidy.items()}
print("\nMax years by statement (tidy):", years)
if any(y is not None for y in years.values()):
    capped = {k: (y is not None and y <= 2024) for k, y in years.items()}
    print("Issue reproduced (any capped at <= 2024):", any(capped.values()))
else:
    print("No data available to confirm reproduction.")

Current Working Directory: /mnt/batch/tasks/shared/LS_root/mounts/clusters/spectral-nature3/code
['common', 'plots', 'stock_fundamental', 'user_specific']
Running run_quarterly_comparison for STX (since=None, plot=False)...
Dataset "us-income-quarterly" not on disk.
- Downloading ... 100.0%
- Extracting zip-file ... Done!
- Loading from disk ... Done!
Dataset "us-balance-quarterly" not on disk.
- Downloading ... 100.0%
- Extracting zip-file ... Done!
- Loading from disk ... Done!
Dataset "us-cashflow-quarterly" not on disk.
- Downloading ... 100.0%
- Extracting zip-file ... Done!
- Loading from disk ... Done!

===== Income Statement (tidy) =====
Rows: 60 | Max Report Date: 2024-09-30 00:00:00 | Max Year: 2024
Last YearQ values: 2023Q4, 2023Q1, 2023Q2, 2024Q3, 2024Q4, 2024Q1


,Report Date,YearQ,Ticker,Metric,Value
0,2023-12-31,2023Q2,STX,Operating Income,9.300000e+07
1,2024-03-31,2024Q3,STX,Net Income,2.500000e+07
2,2024-03-31,2024Q3,STX,Operating Income,1.450000e+08
3,2024-03-31,2024Q3,STX,Revenue,1.655000e+09
4,2024-06-30,2024Q4,STX,Net Income,5.130000e+08
5,2024-06-30,2024Q4,STX,Revenue,1.887000e+09
6,2024-06-30,2024Q4,STX,Operating Income,3.110000e+08
7,2024-09-30,2024Q1,STX,Operating Income,4.040000e+08
8,2024-09-30,2024Q1,STX,Revenue,2.168000e+09
9,2024-09-30,2024Q1,STX,Net Income,3.030000e+08



===== Balance Sheet (tidy) =====
Rows: 60 | Max Report Date: 2024-09-30 00:00:00 | Max Year: 2024
Last YearQ values: 2023Q4, 2023Q1, 2023Q2, 2024Q3, 2024Q4, 2024Q1


,Report Date,YearQ,Ticker,Metric,Value
0,2023-12-31,2023Q2,STX,Total Liabilities,8.963000e+09
1,2024-03-31,2024Q3,STX,Total Equity,-1.889000e+09
2,2024-03-31,2024Q3,STX,Total Liabilities,8.985000e+09
3,2024-03-31,2024Q3,STX,Total Assets,7.096000e+09
4,2024-06-30,2024Q4,STX,Total Equity,-1.491000e+09
5,2024-06-30,2024Q4,STX,Total Assets,7.739000e+09
6,2024-06-30,2024Q4,STX,Total Liabilities,9.230000e+09
7,2024-09-30,2024Q1,STX,Total Liabilities,9.272000e+09
8,2024-09-30,2024Q1,STX,Total Assets,7.972000e+09
9,2024-09-30,2024Q1,STX,Total Equity,-1.300000e+09



===== Cash Flow (tidy) =====
Rows: 60 | Max Report Date: 2024-09-30 00:00:00 | Max Year: 2024
Last YearQ values: 2023Q4, 2023Q1, 2023Q2, 2024Q3, 2024Q4, 2024Q1


...


----- Balance (raw) (raw SimFin subset for STX) -----
Rows (all): 20 | Max Report Date (all): 2024-09-30 00:00:00
Rows (quarters only): 20 | Max Report Date (quarters): 2024-09-30 00:00:00


,Report Date,Fiscal Year,Fiscal Period
0,2023-06-30,2023,Q4
1,2023-09-30,2024,Q1
2,2023-12-31,2024,Q2
3,2024-03-31,2024,Q3
4,2024-06-30,2024,Q4
5,2024-09-30,2025,Q1



----- Cash Flow (raw) (raw SimFin subset for STX) -----
Rows (all): 20 | Max Report Date (all): 2024-09-30 00:00:00
Rows (quarters only): 20 | Max Report Date (quarters): 2024-09-30 00:00:00


,Report Date,Fiscal Year,Fiscal Period
0,2023-06-30,2023,Q4
1,2023-09-30,2024,Q1
2,2023-12-31,2024,Q2
3,2024-03-31,2024,Q3
4,2024-06-30,2024,Q4
5,2024-09-30,2025,Q1



Max years by statement (tidy): {'income': 2024, 'balance': 2024, 'cashflow': 2024}
Issue reproduced (any capped at <= 2024): True


In [2]:
# Core SimFin isolation for STX: load raw quarterly datasets and inspect latest entries

import os
import pandas as pd
import simfin as sf
from IPython.display import display

# If LinkedAuth not desired here, set API key directly (empty string if already cached on disk)
# sf.set_api_key(os.getenv("SIMFIN_API_KEY", ""))  # uncomment and ensure key if needed

# Use same data dir as FundamentalAnalyzer.py (will download if missing)
sf.set_data_dir('../data/stock_fundamental/')

ticker = "STX"

def load_raw_quarterly():
    inc = sf.load(dataset='income',   variant='quarterly', market='us')
    bal = sf.load(dataset='balance',  variant='quarterly', market='us')
    cfs = sf.load(dataset='cashflow', variant='quarterly', market='us')
    return inc, bal, cfs

def extract(df, tk):
    # Works for MultiIndex (SimFin format)
    if isinstance(df.index, pd.MultiIndex) and 'Ticker' in [n for n in df.index.names]:
        return df.xs(tk, level='Ticker', drop_level=False).reset_index()
    # Fallback
    base = df.reset_index()
    return base[base.get('Ticker', '').astype(str) == tk]

inc_raw, bal_raw, cfs_raw = load_raw_quarterly()

frames = {
    "Income": extract(inc_raw, ticker),
    "Balance": extract(bal_raw, ticker),
    "CashFlow": extract(cfs_raw, ticker)
}

for name, df in frames.items():
    print(f"\n=== {name} raw quarterly for {ticker} ===")
    if df.empty:
        print("No data.")
        continue
    df['Report Date'] = pd.to_datetime(df['Report Date'], errors='coerce')
    cols_show = [c for c in ['Report Date','Fiscal Year','Fiscal Period'] if c in df.columns]
    tail = df.sort_values('Report Date').tail(8)[cols_show].reset_index(drop=True)
    display(tail)
    max_report = df['Report Date'].max()
    max_fy = df['Fiscal Year'].max()
    print(f"Max Report Date: {max_report} | Max Fiscal Year: {max_fy}")

# Show mapping (Report Date year vs Fiscal Year) to illustrate discrepancy
print("\n--- Year mapping (Report Date year vs Fiscal Year) ---")
map_rows = (
    frames["Income"]
    .sort_values('Report Date')[['Report Date','Fiscal Year','Fiscal Period']]
    .tail(8)
    .copy()
)
map_rows['Calendar Year'] = map_rows['Report Date'].dt.year
display(map_rows.reset_index(drop=True))

# Explanation of cutoff behavior:
print(
    "\nNote: The latest row shows Fiscal Year advancing to 2025 while Report Date is still in 2024 (Seagate's FY ends in late Sep). "
    "If downstream code constructs YearQ using Report Date.year + Quarter, the 2025 Q1 will appear as 2024Q1, capping visible data at 2024."
)

Dataset "us-income-quarterly" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-balance-quarterly" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-quarterly" on disk (0 days old).
- Loading from disk ... Done!

=== Income raw quarterly for STX ===


,Report Date,Fiscal Year,Fiscal Period
0,2022-12-31,2023,Q2
1,2023-03-31,2023,Q3
2,2023-06-30,2023,Q4
3,2023-09-30,2024,Q1
4,2023-12-31,2024,Q2
5,2024-03-31,2024,Q3
6,2024-06-30,2024,Q4
7,2024-09-30,2025,Q1


Max Report Date: 2024-09-30 00:00:00 | Max Fiscal Year: 2025

=== Balance raw quarterly for STX ===


,Report Date,Fiscal Year,Fiscal Period
0,2022-12-31,2023,Q2
1,2023-03-31,2023,Q3
2,2023-06-30,2023,Q4
3,2023-09-30,2024,Q1
4,2023-12-31,2024,Q2
5,2024-03-31,2024,Q3
6,2024-06-30,2024,Q4
7,2024-09-30,2025,Q1


Max Report Date: 2024-09-30 00:00:00 | Max Fiscal Year: 2025

=== CashFlow raw quarterly for STX ===


,Report Date,Fiscal Year,Fiscal Period
0,2022-12-31,2023,Q2
1,2023-03-31,2023,Q3
2,2023-06-30,2023,Q4
3,2023-09-30,2024,Q1
4,2023-12-31,2024,Q2
5,2024-03-31,2024,Q3
6,2024-06-30,2024,Q4
7,2024-09-30,2025,Q1


Max Report Date: 2024-09-30 00:00:00 | Max Fiscal Year: 2025

--- Year mapping (Report Date year vs Fiscal Year) ---


,Report Date,Fiscal Year,Fiscal Period,Calendar Year
0,2022-12-31,2023,Q2,2022
1,2023-03-31,2023,Q3,2023
2,2023-06-30,2023,Q4,2023
3,2023-09-30,2024,Q1,2023
4,2023-12-31,2024,Q2,2023
5,2024-03-31,2024,Q3,2024
6,2024-06-30,2024,Q4,2024
7,2024-09-30,2025,Q1,2024



Note: The latest row shows Fiscal Year advancing to 2025 while Report Date is still in 2024 (Seagate's FY ends in late Sep). If downstream code constructs YearQ using Report Date.year + Quarter, the 2025 Q1 will appear as 2024Q1, capping visible data at 2024.


In [3]:
# Show raw SimFin quarterly tables for STX (no shaping)

import os
from pathlib import Path
import pandas as pd
import simfin as sf
from IPython.display import display

# Wider display to see all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Try to set API key from LinkedAuth if available (falls back to disk cache)
try:
    import LinkedAuth
    token = LinkedAuth.get_creds("spectral-nature-kvault", retreive=['SimFinAPI'])[0]
    if token:
        sf.set_api_key(token)
except Exception:
    pass

# Use an absolute data dir to avoid surprises with relative paths
data_dir = Path('../data/stock_fundamental').resolve()
data_dir.mkdir(parents=True, exist_ok=True)
sf.set_data_dir(str(data_dir))
print("SimFin data dir:", sf.get_data_dir())

# Load raw quarterly datasets
inc = sf.load(dataset='income',   variant='quarterly', market='us')
bal = sf.load(dataset='balance',  variant='quarterly', market='us')
cfs = sf.load(dataset='cashflow', variant='quarterly', market='us')

def filter_ticker(df, ticker):
    if isinstance(df.index, pd.MultiIndex) and 'Ticker' in list(df.index.names):
        try:
            return df.xs(ticker, level='Ticker', drop_level=False).reset_index()
        except KeyError:
            return df.iloc[0:0].reset_index()
    base = df.reset_index()
    col = 'Ticker' if 'Ticker' in base.columns else ('ticker' if 'ticker' in base.columns else None)
    return base[base[col].astype(str).eq(ticker)] if col else base.iloc[0:0]

for name, raw in [('Income', inc), ('Balance', bal), ('CashFlow', cfs)]:
    sub = filter_ticker(raw, 'STX')
    print(f"\n=== {name} raw quarterly for STX ===")
    print(f"Rows: {len(sub)}")
    if not sub.empty and 'Report Date' in sub.columns:
        sub['Report Date'] = pd.to_datetime(sub['Report Date'], errors='coerce')
        print(f"Report Date range: {sub['Report Date'].min()} -> {sub['Report Date'].max()}")
    print("Columns:", list(sub.columns))
    display(sub)
    

SimFin data dir: /mnt/batch/tasks/shared/LS_root/mounts/clusters/spectral-nature3/data/stock_fundamental
Dataset "us-income-quarterly" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-balance-quarterly" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-quarterly" on disk (0 days old).
- Loading from disk ... Done!

=== Income raw quarterly for STX ===
Rows: 20
Report Date range: 2019-12-31 00:00:00 -> 2024-09-30 00:00:00
Columns: ['index', 'Ticker', 'SimFinId', 'Currency', 'Fiscal Year', 'Fiscal Period', 'Report Date', 'Publish Date', 'Restated Date', 'Shares (Basic)', 'Shares (Diluted)', 'Revenue', 'Cost of Revenue', 'Gross Profit', 'Operating Expenses', 'Selling, General & Administrative', 'Research & Development', 'Depreciation & Amortization', 'Operating Income (Loss)', 'Non-Operating Income (Loss)', 'Interest Expense, Net', 'Pretax Income (Loss), Adj.', 'Abnormal Gains (Losses)', 'Pretax Income (Loss)', 'Income Tax (Expense) Benefit, Net', 'In

,index,Ticker,SimFinId,Currency,Fiscal Year,Fiscal Period,Report Date,Publish Date,Restated Date,Shares (Basic),Shares (Diluted),Revenue,Cost of Revenue,Gross Profit,Operating Expenses,"Selling, General & Administrative",Research & Development,Depreciation & Amortization,Operating Income (Loss),Non-Operating Income (Loss),"Interest Expense, Net","Pretax Income (Loss), Adj.",Abnormal Gains (Losses),Pretax Income (Loss),"Income Tax (Expense) Benefit, Net",Income (Loss) from Continuing Operations,Net Extraordinary Gains (Losses),Net Income,Net Income (Common)
44905,44905,STX,378212,USD,2020,Q2,2019-12-31,2020-02-05,2021-01-28,262000000.0,265000000.0,2.696000e+09,-1.938000e+09,758000000.0,-3.740000e+08,-120000000.0,-250000000.0,-4000000.0,384000000.0,-48000000.0,-44000000.0,336000000.0,NaN,336000000,-18000000.0,318000000,NaN,318000000,318000000
44906,44906,STX,378212,USD,2020,Q3,2020-03-31,2020-04-30,2021-04-29,261000000.0,263000000.0,2.718000e+09,-1.972000e+09,746000000.0,-3.680000e+08,-119000000.0,-246000000.0,-3000000.0,378000000.0,-38000000.0,-45000000.0,340000000.0,-2000000.0,338000000,-18000000.0,320000000,NaN,320000000,320000000
44907,44907,STX,378212,USD,2020,Q4,2020-06-30,2020-08-07,2021-04-29,259000000.0,262000000.0,2.517000e+09,-1.850000e+09,667000000.0,-3.370000e+08,-112000000.0,-222000000.0,-3000000.0,330000000.0,-107000000.0,-48000000.0,223000000.0,-63000000.0,160000000,6000000.0,166000000,NaN,166000000,166000000
44908,44908,STX,378212,USD,2021,Q1,2020-09-30,2020-10-29,2021-10-28,257000000.0,259000000.0,2.314000e+09,-1.718000e+09,596000000.0,-3.440000e+08,-118000000.0,-223000000.0,-3000000.0,252000000.0,-30000000.0,-49000000.0,222000000.0,-1000000.0,221000000,2000000.0,223000000,NaN,223000000,223000000
44909,44909,STX,378212,USD,2021,Q2,2020-12-31,2021-01-28,2022-01-27,249000000.0,251000000.0,2.623000e+09,-1.927000e+09,696000000.0,-3.460000e+08,-122000000.0,-221000000.0,-3000000.0,350000000.0,-57000000.0,-52000000.0,293000000.0,-2000000.0,291000000,-11000000.0,280000000,NaN,280000000,280000000
44910,44910,STX,378212,USD,2021,Q3,2021-03-31,2021-04-29,2022-04-28,233000000.0,237000000.0,2.731000e+09,-1.991000e+09,740000000.0,-3.560000e+08,-126000000.0,-227000000.0,-3000000.0,384000000.0,-47000000.0,-58000000.0,337000000.0,2000000.0,339000000,-10000000.0,329000000,NaN,329000000,329000000
44911,44911,STX,378212,USD,2021,Q4,2021-06-30,2021-08-06,2022-04-28,246000000.0,249000000.0,3.013000e+09,NaN,NaN,-8.135000e+09,-136000000.0,-232000000.0,-3000000.0,514000000.0,-10000000.0,-59000000.0,504000000.0,-7000000.0,497000000,-15000000.0,482000000,NaN,482000000,482000000
44912,44912,STX,378212,USD,2022,Q1,2021-09-30,2021-10-28,2022-10-27,226000000.0,231000000.0,3.115000e+09,-2.159000e+09,956000000.0,-3.690000e+08,-133000000.0,-233000000.0,-3000000.0,587000000.0,-53000000.0,-59000000.0,534000000.0,-1000000.0,533000000,-7000000.0,526000000,NaN,526000000,526000000
44913,44913,STX,378212,USD,2022,Q2,2021-12-31,2022-01-27,2023-01-25,221000000.0,225000000.0,3.116000e+09,-2.168000e+09,948000000.0,-3.670000e+08,-136000000.0,-228000000.0,-3000000.0,581000000.0,-66000000.0,-61000000.0,515000000.0,-1000000.0,514000000,-13000000.0,501000000,NaN,501000000,501000000
44914,44914,STX,378212,USD,2022,Q3,2022-03-31,2022-04-28,2023-04-26,221000000.0,225000000.0,2.802000e+09,-1.996000e+09,806000000.0,-3.770000e+08,-141000000.0,-233000000.0,-3000000.0,429000000.0,-78000000.0,-63000000.0,351000000.0,NaN,351000000,-5000000.0,346000000,NaN,346000000,346000000



=== Balance raw quarterly for STX ===
Rows: 20
Report Date range: 2019-12-31 00:00:00 -> 2024-09-30 00:00:00
Columns: ['index', 'Ticker', 'SimFinId', 'Currency', 'Fiscal Year', 'Fiscal Period', 'Report Date', 'Publish Date', 'Restated Date', 'Shares (Basic)', 'Shares (Diluted)', 'Cash, Cash Equivalents & Short Term Investments', 'Accounts & Notes Receivable', 'Inventories', 'Total Current Assets', 'Property, Plant & Equipment, Net', 'Long Term Investments & Receivables', 'Other Long Term Assets', 'Total Noncurrent Assets', 'Total Assets', 'Payables & Accruals', 'Short Term Debt', 'Total Current Liabilities', 'Long Term Debt', 'Total Noncurrent Liabilities', 'Total Liabilities', 'Share Capital & Additional Paid-In Capital', 'Treasury Stock', 'Retained Earnings', 'Total Equity', 'Total Liabilities & Equity']


,index,Ticker,SimFinId,Currency,Fiscal Year,Fiscal Period,Report Date,Publish Date,Restated Date,Shares (Basic),Shares (Diluted),"Cash, Cash Equivalents & Short Term Investments",Accounts & Notes Receivable,Inventories,Total Current Assets,"Property, Plant & Equipment, Net",Long Term Investments & Receivables,Other Long Term Assets,Total Noncurrent Assets,Total Assets,Payables & Accruals,Short Term Debt,Total Current Liabilities,Long Term Debt,Total Noncurrent Liabilities,Total Liabilities,Share Capital & Additional Paid-In Capital,Treasury Stock,Retained Earnings,Total Equity,Total Liabilities & Equity
44908,44908,STX,378212,USD,2020,Q2,2019-12-31,2020-02-05,2020-02-05,262000000.0,265000000.0,1.744000e+09,1.112000e+09,1.148000e+09,4.152000e+09,2.049000e+09,NaN,2.731000e+09,4.780000e+09,8932000000,2.694000e+09,6.000000e+06,2.700000e+09,4.135000e+09,4.402000e+09,7.102000e+09,6.667000e+09,NaN,-4.804000e+09,1.830000e+09,8932000000
44909,44909,STX,378212,USD,2020,Q3,2020-03-31,2020-04-30,2020-04-30,261000000.0,263000000.0,1.612000e+09,1.160000e+09,1.102000e+09,4.015000e+09,2.093000e+09,NaN,2.721000e+09,4.814000e+09,8829000000,2.678000e+09,1.200000e+07,2.690000e+09,4.091000e+09,4.347000e+09,7.037000e+09,6.725000e+09,NaN,-4.866000e+09,1.792000e+09,8829000000
44910,44910,STX,378212,USD,2020,Q4,2020-06-30,2020-08-07,2021-08-06,259000000.0,262000000.0,1.722000e+09,1.115000e+09,1.142000e+09,4.114000e+09,2.129000e+09,NaN,2.687000e+09,4.816000e+09,8930000000,2.703000e+09,1.900000e+07,2.722000e+09,4.156000e+09,4.421000e+09,7.143000e+09,6.757000e+09,NaN,-4.904000e+09,1.787000e+09,8930000000
44911,44911,STX,378212,USD,2021,Q1,2020-09-30,2020-10-29,2020-10-29,257000000.0,259000000.0,1.664000e+09,8.660000e+08,1.323000e+09,3.994000e+09,2.167000e+09,NaN,2.701000e+09,4.868000e+09,8862000000,2.619000e+09,2.500000e+07,2.644000e+09,4.138000e+09,4.397000e+09,7.041000e+09,6.814000e+09,NaN,-4.947000e+09,1.821000e+09,8862000000
44912,44912,STX,378212,USD,2021,Q2,2020-12-31,2021-01-28,2021-01-28,249000000.0,251000000.0,1.799000e+09,8.010000e+08,1.318000e+09,4.081000e+09,2.218000e+09,NaN,2.687000e+09,4.905000e+09,8986000000,2.596000e+09,2.500000e+07,2.621000e+09,5.120000e+09,5.375000e+09,7.996000e+09,6.855000e+09,NaN,-5.829000e+09,9.900000e+08,8986000000
44913,44913,STX,378212,USD,2021,Q3,2021-03-31,2021-04-29,2021-04-29,233000000.0,237000000.0,1.212000e+09,9.780000e+08,1.281000e+09,3.692000e+09,2.215000e+09,NaN,2.697000e+09,4.912000e+09,8604000000,2.748000e+09,2.450000e+08,2.993000e+09,4.897000e+09,5.127000e+09,8.120000e+09,6.939000e+09,NaN,-6.417000e+09,4.840000e+08,8604000000
44914,44914,STX,378212,USD,2021,Q4,2021-06-30,2021-08-06,2022-08-05,246000000.0,249000000.0,1.209000e+09,1.158000e+09,1.204000e+09,3.779000e+09,2.181000e+09,NaN,2.715000e+09,4.896000e+09,8675000000,2.676000e+09,2.450000e+08,2.921000e+09,4.894000e+09,5.123000e+09,8.044000e+09,6.977000e+09,NaN,-6.305000e+09,6.310000e+08,8675000000
44915,44915,STX,378212,USD,2022,Q1,2021-09-30,2021-10-28,2021-10-28,226000000.0,231000000.0,9.910000e+08,1.301000e+09,1.188000e+09,3.668000e+09,2.213000e+09,NaN,2.732000e+09,4.945000e+09,8613000000,2.644000e+09,2.450000e+08,2.889000e+09,4.891000e+09,5.123000e+09,8.012000e+09,NaN,NaN,-6.398000e+09,6.010000e+08,8613000000
44916,44916,STX,378212,USD,2022,Q2,2021-12-31,2022-01-27,2022-01-27,221000000.0,225000000.0,1.535000e+09,1.399000e+09,1.287000e+09,4.450000e+09,2.216000e+09,NaN,2.709000e+09,4.925000e+09,9375000000,2.757000e+09,2.350000e+08,2.992000e+09,5.626000e+09,5.857000e+09,8.849000e+09,NaN,NaN,-6.533000e+09,5.260000e+08,9375000000
44917,44917,STX,378212,USD,2022,Q3,2022-03-31,2022-04-28,2022-04-28,221000000.0,225000000.0,1.138000e+09,1.344000e+09,1.479000e+09,4.259000e+09,2.197000e+09,NaN,2.689000e+09,4.886000e+09,9145000000,2.851000e+09,3.000000e+07,2.881000e+09,5.614000e+09,5.843000e+09,8.724000e+09,NaN,NaN,-6.767000e+09,4.210000e+08,9145000000



=== CashFlow raw quarterly for STX ===
Rows: 20
Report Date range: 2019-12-31 00:00:00 -> 2024-09-30 00:00:00
Columns: ['index', 'Ticker', 'SimFinId', 'Currency', 'Fiscal Year', 'Fiscal Period', 'Report Date', 'Publish Date', 'Restated Date', 'Shares (Basic)', 'Shares (Diluted)', 'Net Income/Starting Line', 'Depreciation & Amortization', 'Non-Cash Items', 'Change in Working Capital', 'Change in Accounts Receivable', 'Change in Inventories', 'Change in Accounts Payable', 'Change in Other', 'Net Cash from Operating Activities', 'Change in Fixed Assets & Intangibles', 'Net Change in Long Term Investment', 'Net Cash from Acquisitions & Divestitures', 'Net Cash from Investing Activities', 'Dividends Paid', 'Cash from (Repayment of) Debt', 'Cash from (Repurchase of) Equity', 'Net Cash from Financing Activities', 'Net Change in Cash']


,index,Ticker,SimFinId,Currency,Fiscal Year,Fiscal Period,Report Date,Publish Date,Restated Date,Shares (Basic),Shares (Diluted),Net Income/Starting Line,Depreciation & Amortization,Non-Cash Items,Change in Working Capital,Change in Accounts Receivable,Change in Inventories,Change in Accounts Payable,Change in Other,Net Cash from Operating Activities,Change in Fixed Assets & Intangibles,Net Change in Long Term Investment,Net Cash from Acquisitions & Divestitures,Net Cash from Investing Activities,Dividends Paid,Cash from (Repayment of) Debt,Cash from (Repurchase of) Equity,Net Cash from Financing Activities,Net Change in Cash
44908,44908,STX,378212,USD,2020,Q2,2019-12-31,2020-02-05,2020-10-29,262000000.0,265000000.0,3.180000e+08,93000000.0,38000000.0,31000000.0,NaN,NaN,NaN,NaN,480000000,-193000000.0,-41000000.0,NaN,-234000000.0,-165000000.0,0.0,-120000000.0,-2.890000e+08,-40000000
44909,44909,STX,378212,USD,2020,Q3,2020-03-31,2020-04-30,2021-01-28,261000000.0,263000000.0,3.200000e+08,94000000.0,42000000.0,-66000000.0,NaN,NaN,NaN,NaN,390000000,-130000000.0,-12000000.0,NaN,-142000000.0,-170000000.0,-40000000.0,-164000000.0,-3.740000e+08,-132000000
44910,44910,STX,378212,USD,2020,Q4,2020-06-30,2020-08-07,2021-04-29,259000000.0,262000000.0,1.660000e+08,100000000.0,75000000.0,47000000.0,NaN,NaN,NaN,NaN,388000000,-114000000.0,6000000.0,NaN,-108000000.0,-168000000.0,44000000.0,-52000000.0,-1.770000e+08,110000000
44911,44911,STX,378212,USD,2021,Q1,2020-09-30,2020-10-29,2021-10-28,257000000.0,259000000.0,2.230000e+08,99000000.0,2000000.0,-27000000.0,NaN,NaN,NaN,NaN,297000000,-111000000.0,7000000.0,NaN,-104000000.0,-167000000.0,-13000000.0,-39000000.0,-2.510000e+08,-58000000
44912,44912,STX,378212,USD,2021,Q2,2020-12-31,2021-01-28,2021-10-28,249000000.0,251000000.0,2.800000e+08,96000000.0,47000000.0,50000000.0,NaN,NaN,NaN,NaN,473000000,-159000000.0,0.0,NaN,-159000000.0,-167000000.0,992000000.0,-989000000.0,-1.790000e+08,135000000
44913,44913,STX,378212,USD,2021,Q3,2021-03-31,2021-04-29,2022-01-27,233000000.0,237000000.0,3.290000e+08,99000000.0,23000000.0,-73000000.0,NaN,NaN,NaN,NaN,378000000,-100000000.0,0.0,NaN,-97000000.0,-161000000.0,-6000000.0,-696000000.0,-8.680000e+08,-587000000
44914,44914,STX,378212,USD,2021,Q4,2021-06-30,2021-08-06,2022-04-28,246000000.0,249000000.0,4.820000e+08,103000000.0,-13000000.0,-94000000.0,NaN,NaN,NaN,NaN,478000000,-124000000.0,18000000.0,NaN,-106000000.0,-154000000.0,-6000000.0,-215000000.0,-3.750000e+08,-3000000
44915,44915,STX,378212,USD,2022,Q1,2021-09-30,2021-10-28,2022-10-27,226000000.0,231000000.0,5.260000e+08,104000000.0,32000000.0,-166000000.0,NaN,NaN,NaN,NaN,496000000,-117000000.0,-3000000.0,NaN,-120000000.0,-153000000.0,-6000000.0,-392000000.0,-5.940000e+08,-218000000
44916,44916,STX,378212,USD,2022,Q2,2021-12-31,2022-01-27,2022-10-27,221000000.0,225000000.0,-5.300000e+08,179000000.0,-150000000.0,501000000.0,NaN,NaN,NaN,NaN,0,-92000000.0,2000000.0,NaN,-90000000.0,-139000000.0,606000000.0,13000000.0,4.630000e+08,373000000
44917,44917,STX,378212,USD,2022,Q3,2022-03-31,2022-04-28,2024-01-26,221000000.0,225000000.0,1.377000e+09,41000000.0,272000000.0,-709000000.0,NaN,NaN,NaN,NaN,981000000,-100000000.0,17000000.0,NaN,-83000000.0,-166000000.0,-101000000.0,-866000000.0,-1.124000e+09,-226000000


In [5]:
# Minimal: load SimFin quarterly datasets and display raw STX rows

import pandas as pd
import simfin as sf
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

sf.set_data_dir('../data/stock_fundamental')

inc = sf.load(dataset='income',   variant='quarterly', market='us')
bal = sf.load(dataset='balance',  variant='quarterly', market='us')
cfs = sf.load(dataset='cashflow', variant='quarterly', market='us')

def stx(df):
    base = df.reset_index()
    col = 'Ticker' if 'Ticker' in base.columns else ('ticker' if 'ticker' in base.columns else None)
    return base[base[col].astype(str).eq('STX')] if col else base

inc_stx = stx(inc)
bal_stx = stx(bal)
cfs_stx = stx(cfs)

for name, df in [('Income', inc_stx), ('Balance', bal_stx), ('CashFlow', cfs_stx)]:
    print(f"\n=== {name} raw quarterly for STX ===")
    if 'Report Date' in df.columns:
        df['Report Date'] = pd.to_datetime(df['Report Date'], errors='coerce')
        print(f"Report Date range: {df['Report Date'].min()} -> {df['Report Date'].max()}")
    display(df)

Dataset "us-income-quarterly" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-balance-quarterly" on disk (0 days old).
- Loading from disk ... Done!
Dataset "us-cashflow-quarterly" on disk (0 days old).
- Loading from disk ... Done!

=== Income raw quarterly for STX ===
Report Date range: 2019-12-31 00:00:00 -> 2024-09-30 00:00:00


,index,Ticker,SimFinId,Currency,Fiscal Year,Fiscal Period,Report Date,Publish Date,Restated Date,Shares (Basic),Shares (Diluted),Revenue,Cost of Revenue,Gross Profit,Operating Expenses,"Selling, General & Administrative",Research & Development,Depreciation & Amortization,Operating Income (Loss),Non-Operating Income (Loss),"Interest Expense, Net","Pretax Income (Loss), Adj.",Abnormal Gains (Losses),Pretax Income (Loss),"Income Tax (Expense) Benefit, Net",Income (Loss) from Continuing Operations,Net Extraordinary Gains (Losses),Net Income,Net Income (Common)
44905,44905,STX,378212,USD,2020,Q2,2019-12-31,2020-02-05,2021-01-28,262000000.0,265000000.0,2.696000e+09,-1.938000e+09,758000000.0,-3.740000e+08,-120000000.0,-250000000.0,-4000000.0,384000000.0,-48000000.0,-44000000.0,336000000.0,NaN,336000000,-18000000.0,318000000,NaN,318000000,318000000
44906,44906,STX,378212,USD,2020,Q3,2020-03-31,2020-04-30,2021-04-29,261000000.0,263000000.0,2.718000e+09,-1.972000e+09,746000000.0,-3.680000e+08,-119000000.0,-246000000.0,-3000000.0,378000000.0,-38000000.0,-45000000.0,340000000.0,-2000000.0,338000000,-18000000.0,320000000,NaN,320000000,320000000
44907,44907,STX,378212,USD,2020,Q4,2020-06-30,2020-08-07,2021-04-29,259000000.0,262000000.0,2.517000e+09,-1.850000e+09,667000000.0,-3.370000e+08,-112000000.0,-222000000.0,-3000000.0,330000000.0,-107000000.0,-48000000.0,223000000.0,-63000000.0,160000000,6000000.0,166000000,NaN,166000000,166000000
44908,44908,STX,378212,USD,2021,Q1,2020-09-30,2020-10-29,2021-10-28,257000000.0,259000000.0,2.314000e+09,-1.718000e+09,596000000.0,-3.440000e+08,-118000000.0,-223000000.0,-3000000.0,252000000.0,-30000000.0,-49000000.0,222000000.0,-1000000.0,221000000,2000000.0,223000000,NaN,223000000,223000000
44909,44909,STX,378212,USD,2021,Q2,2020-12-31,2021-01-28,2022-01-27,249000000.0,251000000.0,2.623000e+09,-1.927000e+09,696000000.0,-3.460000e+08,-122000000.0,-221000000.0,-3000000.0,350000000.0,-57000000.0,-52000000.0,293000000.0,-2000000.0,291000000,-11000000.0,280000000,NaN,280000000,280000000
44910,44910,STX,378212,USD,2021,Q3,2021-03-31,2021-04-29,2022-04-28,233000000.0,237000000.0,2.731000e+09,-1.991000e+09,740000000.0,-3.560000e+08,-126000000.0,-227000000.0,-3000000.0,384000000.0,-47000000.0,-58000000.0,337000000.0,2000000.0,339000000,-10000000.0,329000000,NaN,329000000,329000000
44911,44911,STX,378212,USD,2021,Q4,2021-06-30,2021-08-06,2022-04-28,246000000.0,249000000.0,3.013000e+09,NaN,NaN,-8.135000e+09,-136000000.0,-232000000.0,-3000000.0,514000000.0,-10000000.0,-59000000.0,504000000.0,-7000000.0,497000000,-15000000.0,482000000,NaN,482000000,482000000
44912,44912,STX,378212,USD,2022,Q1,2021-09-30,2021-10-28,2022-10-27,226000000.0,231000000.0,3.115000e+09,-2.159000e+09,956000000.0,-3.690000e+08,-133000000.0,-233000000.0,-3000000.0,587000000.0,-53000000.0,-59000000.0,534000000.0,-1000000.0,533000000,-7000000.0,526000000,NaN,526000000,526000000
44913,44913,STX,378212,USD,2022,Q2,2021-12-31,2022-01-27,2023-01-25,221000000.0,225000000.0,3.116000e+09,-2.168000e+09,948000000.0,-3.670000e+08,-136000000.0,-228000000.0,-3000000.0,581000000.0,-66000000.0,-61000000.0,515000000.0,-1000000.0,514000000,-13000000.0,501000000,NaN,501000000,501000000
44914,44914,STX,378212,USD,2022,Q3,2022-03-31,2022-04-28,2023-04-26,221000000.0,225000000.0,2.802000e+09,-1.996000e+09,806000000.0,-3.770000e+08,-141000000.0,-233000000.0,-3000000.0,429000000.0,-78000000.0,-63000000.0,351000000.0,NaN,351000000,-5000000.0,346000000,NaN,346000000,346000000



=== Balance raw quarterly for STX ===
Report Date range: 2019-12-31 00:00:00 -> 2024-09-30 00:00:00


,index,Ticker,SimFinId,Currency,Fiscal Year,Fiscal Period,Report Date,Publish Date,Restated Date,Shares (Basic),Shares (Diluted),"Cash, Cash Equivalents & Short Term Investments",Accounts & Notes Receivable,Inventories,Total Current Assets,"Property, Plant & Equipment, Net",Long Term Investments & Receivables,Other Long Term Assets,Total Noncurrent Assets,Total Assets,Payables & Accruals,Short Term Debt,Total Current Liabilities,Long Term Debt,Total Noncurrent Liabilities,Total Liabilities,Share Capital & Additional Paid-In Capital,Treasury Stock,Retained Earnings,Total Equity,Total Liabilities & Equity
44908,44908,STX,378212,USD,2020,Q2,2019-12-31,2020-02-05,2020-02-05,262000000.0,265000000.0,1.744000e+09,1.112000e+09,1.148000e+09,4.152000e+09,2.049000e+09,NaN,2.731000e+09,4.780000e+09,8932000000,2.694000e+09,6.000000e+06,2.700000e+09,4.135000e+09,4.402000e+09,7.102000e+09,6.667000e+09,NaN,-4.804000e+09,1.830000e+09,8932000000
44909,44909,STX,378212,USD,2020,Q3,2020-03-31,2020-04-30,2020-04-30,261000000.0,263000000.0,1.612000e+09,1.160000e+09,1.102000e+09,4.015000e+09,2.093000e+09,NaN,2.721000e+09,4.814000e+09,8829000000,2.678000e+09,1.200000e+07,2.690000e+09,4.091000e+09,4.347000e+09,7.037000e+09,6.725000e+09,NaN,-4.866000e+09,1.792000e+09,8829000000
44910,44910,STX,378212,USD,2020,Q4,2020-06-30,2020-08-07,2021-08-06,259000000.0,262000000.0,1.722000e+09,1.115000e+09,1.142000e+09,4.114000e+09,2.129000e+09,NaN,2.687000e+09,4.816000e+09,8930000000,2.703000e+09,1.900000e+07,2.722000e+09,4.156000e+09,4.421000e+09,7.143000e+09,6.757000e+09,NaN,-4.904000e+09,1.787000e+09,8930000000
44911,44911,STX,378212,USD,2021,Q1,2020-09-30,2020-10-29,2020-10-29,257000000.0,259000000.0,1.664000e+09,8.660000e+08,1.323000e+09,3.994000e+09,2.167000e+09,NaN,2.701000e+09,4.868000e+09,8862000000,2.619000e+09,2.500000e+07,2.644000e+09,4.138000e+09,4.397000e+09,7.041000e+09,6.814000e+09,NaN,-4.947000e+09,1.821000e+09,8862000000
44912,44912,STX,378212,USD,2021,Q2,2020-12-31,2021-01-28,2021-01-28,249000000.0,251000000.0,1.799000e+09,8.010000e+08,1.318000e+09,4.081000e+09,2.218000e+09,NaN,2.687000e+09,4.905000e+09,8986000000,2.596000e+09,2.500000e+07,2.621000e+09,5.120000e+09,5.375000e+09,7.996000e+09,6.855000e+09,NaN,-5.829000e+09,9.900000e+08,8986000000
44913,44913,STX,378212,USD,2021,Q3,2021-03-31,2021-04-29,2021-04-29,233000000.0,237000000.0,1.212000e+09,9.780000e+08,1.281000e+09,3.692000e+09,2.215000e+09,NaN,2.697000e+09,4.912000e+09,8604000000,2.748000e+09,2.450000e+08,2.993000e+09,4.897000e+09,5.127000e+09,8.120000e+09,6.939000e+09,NaN,-6.417000e+09,4.840000e+08,8604000000
44914,44914,STX,378212,USD,2021,Q4,2021-06-30,2021-08-06,2022-08-05,246000000.0,249000000.0,1.209000e+09,1.158000e+09,1.204000e+09,3.779000e+09,2.181000e+09,NaN,2.715000e+09,4.896000e+09,8675000000,2.676000e+09,2.450000e+08,2.921000e+09,4.894000e+09,5.123000e+09,8.044000e+09,6.977000e+09,NaN,-6.305000e+09,6.310000e+08,8675000000
44915,44915,STX,378212,USD,2022,Q1,2021-09-30,2021-10-28,2021-10-28,226000000.0,231000000.0,9.910000e+08,1.301000e+09,1.188000e+09,3.668000e+09,2.213000e+09,NaN,2.732000e+09,4.945000e+09,8613000000,2.644000e+09,2.450000e+08,2.889000e+09,4.891000e+09,5.123000e+09,8.012000e+09,NaN,NaN,-6.398000e+09,6.010000e+08,8613000000
44916,44916,STX,378212,USD,2022,Q2,2021-12-31,2022-01-27,2022-01-27,221000000.0,225000000.0,1.535000e+09,1.399000e+09,1.287000e+09,4.450000e+09,2.216000e+09,NaN,2.709000e+09,4.925000e+09,9375000000,2.757000e+09,2.350000e+08,2.992000e+09,5.626000e+09,5.857000e+09,8.849000e+09,NaN,NaN,-6.533000e+09,5.260000e+08,9375000000
44917,44917,STX,378212,USD,2022,Q3,2022-03-31,2022-04-28,2022-04-28,221000000.0,225000000.0,1.138000e+09,1.344000e+09,1.479000e+09,4.259000e+09,2.197000e+09,NaN,2.689000e+09,4.886000e+09,9145000000,2.851000e+09,3.000000e+07,2.881000e+09,5.614000e+09,5.843000e+09,8.724000e+09,NaN,NaN,-6.767000e+09,4.210000e+08,9145000000



=== CashFlow raw quarterly for STX ===
Report Date range: 2019-12-31 00:00:00 -> 2024-09-30 00:00:00


,index,Ticker,SimFinId,Currency,Fiscal Year,Fiscal Period,Report Date,Publish Date,Restated Date,Shares (Basic),Shares (Diluted),Net Income/Starting Line,Depreciation & Amortization,Non-Cash Items,Change in Working Capital,Change in Accounts Receivable,Change in Inventories,Change in Accounts Payable,Change in Other,Net Cash from Operating Activities,Change in Fixed Assets & Intangibles,Net Change in Long Term Investment,Net Cash from Acquisitions & Divestitures,Net Cash from Investing Activities,Dividends Paid,Cash from (Repayment of) Debt,Cash from (Repurchase of) Equity,Net Cash from Financing Activities,Net Change in Cash
44908,44908,STX,378212,USD,2020,Q2,2019-12-31,2020-02-05,2020-10-29,262000000.0,265000000.0,3.180000e+08,93000000.0,38000000.0,31000000.0,NaN,NaN,NaN,NaN,480000000,-193000000.0,-41000000.0,NaN,-234000000.0,-165000000.0,0.0,-120000000.0,-2.890000e+08,-40000000
44909,44909,STX,378212,USD,2020,Q3,2020-03-31,2020-04-30,2021-01-28,261000000.0,263000000.0,3.200000e+08,94000000.0,42000000.0,-66000000.0,NaN,NaN,NaN,NaN,390000000,-130000000.0,-12000000.0,NaN,-142000000.0,-170000000.0,-40000000.0,-164000000.0,-3.740000e+08,-132000000
44910,44910,STX,378212,USD,2020,Q4,2020-06-30,2020-08-07,2021-04-29,259000000.0,262000000.0,1.660000e+08,100000000.0,75000000.0,47000000.0,NaN,NaN,NaN,NaN,388000000,-114000000.0,6000000.0,NaN,-108000000.0,-168000000.0,44000000.0,-52000000.0,-1.770000e+08,110000000
44911,44911,STX,378212,USD,2021,Q1,2020-09-30,2020-10-29,2021-10-28,257000000.0,259000000.0,2.230000e+08,99000000.0,2000000.0,-27000000.0,NaN,NaN,NaN,NaN,297000000,-111000000.0,7000000.0,NaN,-104000000.0,-167000000.0,-13000000.0,-39000000.0,-2.510000e+08,-58000000
44912,44912,STX,378212,USD,2021,Q2,2020-12-31,2021-01-28,2021-10-28,249000000.0,251000000.0,2.800000e+08,96000000.0,47000000.0,50000000.0,NaN,NaN,NaN,NaN,473000000,-159000000.0,0.0,NaN,-159000000.0,-167000000.0,992000000.0,-989000000.0,-1.790000e+08,135000000
44913,44913,STX,378212,USD,2021,Q3,2021-03-31,2021-04-29,2022-01-27,233000000.0,237000000.0,3.290000e+08,99000000.0,23000000.0,-73000000.0,NaN,NaN,NaN,NaN,378000000,-100000000.0,0.0,NaN,-97000000.0,-161000000.0,-6000000.0,-696000000.0,-8.680000e+08,-587000000
44914,44914,STX,378212,USD,2021,Q4,2021-06-30,2021-08-06,2022-04-28,246000000.0,249000000.0,4.820000e+08,103000000.0,-13000000.0,-94000000.0,NaN,NaN,NaN,NaN,478000000,-124000000.0,18000000.0,NaN,-106000000.0,-154000000.0,-6000000.0,-215000000.0,-3.750000e+08,-3000000
44915,44915,STX,378212,USD,2022,Q1,2021-09-30,2021-10-28,2022-10-27,226000000.0,231000000.0,5.260000e+08,104000000.0,32000000.0,-166000000.0,NaN,NaN,NaN,NaN,496000000,-117000000.0,-3000000.0,NaN,-120000000.0,-153000000.0,-6000000.0,-392000000.0,-5.940000e+08,-218000000
44916,44916,STX,378212,USD,2022,Q2,2021-12-31,2022-01-27,2022-10-27,221000000.0,225000000.0,-5.300000e+08,179000000.0,-150000000.0,501000000.0,NaN,NaN,NaN,NaN,0,-92000000.0,2000000.0,NaN,-90000000.0,-139000000.0,606000000.0,13000000.0,4.630000e+08,373000000
44917,44917,STX,378212,USD,2022,Q3,2022-03-31,2022-04-28,2024-01-26,221000000.0,225000000.0,1.377000e+09,41000000.0,272000000.0,-709000000.0,NaN,NaN,NaN,NaN,981000000,-100000000.0,17000000.0,NaN,-83000000.0,-166000000.0,-101000000.0,-866000000.0,-1.124000e+09,-226000000


In [6]:
import simfin as sf
from simfin.names import TICKER, REPORT_DATE, PUBLISH_DATE

inc = sf.load(dataset='income',   variant='quarterly', market='us')
def latest_by_ticker(df, n=10):
    x = (df.reset_index()[[TICKER, REPORT_DATE, PUBLISH_DATE]]
           .groupby(TICKER).max().sort_values(REPORT_DATE))
    return x.tail(n)

print(latest_by_ticker(inc, n=25))

Dataset "us-income-quarterly" on disk (0 days old).
- Loading from disk ... Done!
       Report Date Publish Date
Ticker                         
PURE    2024-10-31   2024-12-16
JILL    2024-10-31   2024-12-11
NVDA    2024-10-31   2024-11-20
PVH     2024-10-31   2024-12-09
VALU    2024-10-31   2024-12-13
CRWD    2024-10-31   2024-11-27
SPWH    2024-10-31   2024-12-11
CRDO    2024-10-31   2024-12-03
CSBR    2024-10-31   2024-12-16
GEF     2024-10-31   2024-12-23
JVA     2024-10-31   2025-01-31
JW-A    2024-10-31   2024-12-06
JWN     2024-10-31   2024-12-05
NTNX    2024-10-31   2024-12-05
VEEV    2024-10-31   2024-12-09
KALV    2024-10-31   2024-12-05
NTAP    2024-10-31   2024-11-25
GCO     2024-10-31   2024-12-12
KFY     2024-10-31   2024-12-09
ASO     2024-10-31   2024-12-10
KIRK    2024-10-31   2024-12-06
VIRC    2024-10-31   2024-12-09
TGT     2024-10-31   2024-11-27
CSCO    2024-10-31   2024-11-19
A       2024-10-31   2024-12-20


In [9]:
r.json()

{'error': 'Full authentication is required to access this resource'}

In [8]:
import requests, pandas as pd

BASE = "https://prod.simfin.com/api/v3/companies/statements"
params = {
    "ticker": "AAPL",          # or a list, depending on endpoint rules
    "statements": "pl",        # pl|bs|cf  (income/balance/cashflow)
    "period": "qf",            # quarterly; check docs for exact token
    "asreported": "true",      # if your plan exposes as-reported
    "api-key": token
}
r = requests.get(BASE, params=params, timeout=30)
df = pd.DataFrame(r.json())

ValueError: If using all scalar values, you must pass an index

In [14]:
token

'92ede6c9-c1bb-4975-9418-72fe65761f2c'

In [13]:
# SimFin REST API v3: fetch raw quarterly statements for STX (no try/except)

import requests, pandas as pd

# Use the token you already fetched earlier in the notebook
tok = token[0] if isinstance(token, (list, tuple)) else token

BASE = "https://prod.simfin.com/api/v3/companies/statements"
COMMON = {"ticker": "STX", "period": "qf", "api-key": tok}  # period=qf = quarterly fiscal

def fetch(stmt):
    params = {**COMMON, "statements": stmt}  # stmt in {'pl','bs','cf'}
    j = requests.get(BASE, params=params, timeout=30).json()
    # Normalize both possible shapes
    if isinstance(j, dict) and "data" in j and "columns" in j:
        return pd.DataFrame(j["data"], columns=j["columns"])
    return pd.json_normalize(j)

pl = fetch("pl")   # income
bs = fetch("bs")   # balance
cf = fetch("cf")   # cash flow

# Inspect
pd.set_option("display.max_columns", None); pd.set_option("display.width", 200)
print("Income (pl):", pl.shape); display(pl)
print("Balance (bs):", bs.shape); display(bs)
print("Cash Flow (cf):", cf.shape); display(cf)
# SimFin REST API v3: fetch raw quarterly statements for STX (no try/except)

import requests, pandas as pd

# Use the token you already fetched earlier in the notebook
tok = token[0] if isinstance(token, (list, tuple)) else token

BASE = "https://prod.simfin.com/api/v3/companies/statements"
COMMON = {"ticker": "STX", "period": "qf", "api-key": tok}  # period=qf = quarterly fiscal

def fetch(stmt):
    params = {**COMMON, "statements": stmt}  # stmt in {'pl','bs','cf'}
    j = requests.get(BASE, params=params, timeout=30).json()
    # Normalize both possible shapes
    if isinstance(j, dict) and "data" in j and "columns" in j:
        return pd.DataFrame(j["data"], columns=j["columns"])
    return pd.json_normalize(j)

pl = fetch("pl")   # income
bs = fetch("bs")   # balance
cf = fetch("cf")   # cash flow

# Inspect
pd.set_option("display.max_columns", None); pd.set_option("display.width", 200)
print("Income (pl):", pl.shape); display(pl)
print("Balance (bs):", bs.shape); display(bs)
print("Cash Flow (cf):", cf.shape); display(cf)

Income (pl): (1, 1)


,error
0,Full authentication is required to access this...


Balance (bs): (1, 1)


,error
0,Full authentication is required to access this...


Cash Flow (cf): (1, 1)


...

,error
0,Full authentication is required to access this...


Balance (bs): (1, 1)


,error
0,Full authentication is required to access this...


Cash Flow (cf): (1, 1)


,error
0,Full authentication is required to access this...
